<a href="https://colab.research.google.com/github/haruki-N/DeepLearningBasics/blob/main/DETR_resnet101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount Google Drive and set directories

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')
notebook_dir = '/content/drive'

notebook_dir = '/content/drive/MyDrive/object_detection/DETR_val2017_ResNet101'
base_url = 'http://images.cocodataset.org'
dataset_url = f'{base_url}/zips/val2017.zip'
annotations_url = f'{base_url}/annotations/annotations_trainval2017.val'
dataset_folder = 'val2017'
anno_file = 'instance_val2017.json'
temp_dir = '/content/data'

if not os.path.exists(temp_dir):
  os.makedirs(temp_dir)

dataset_zip_path = os.path.join(temp_dir, 'dataset.zip')
annotations_zip_path = os.path.join(temp_dir, 'annotations.zip')
dataset_extract_path = temp_dir
annotations_extract_path = temp_dir
dataset_path = os.path.join(dataset_extract_path, dataset_folder)
instance_json_path = os.path.join(annotations_extract_path, 'annotations', anno_file)

MessageError: Error: credential propagation was unsuccessful

# Data preparation

In [ ]:
import requests
import time
from zipfile import ZipFile

def download_file(url, dest):
  if os.path.exists(dest):
    print(f'File already exists: {dest}. Skipping download.')
    return

  print(f'Donwloaind {url} ...')
  start_time = time.time()
  response = requests.get(url, stream=True)
  if response.status_code == 200:
    with open(dest, 'wb') as f:
      for chunk in response.iter_content(chunk_size=1024):
        if chunk:
          f.write(chunk)
    elapsed_time = time.time() - start_time
    print(f'Downloaded and saved to {dest} in {elapsed_time:.2f} seconds.')
  else:
    print(f'Failed to download {url} (HTTP {response.status_code})')

def extract_zip(zip_path, extract_to, required_file=None):
  if (
      os.path.exists(extract_to)
      and required_file
      and os.path.exists(os.path.join(extract_to, required_file))
  ):
    print(f'Required file {required_file} already exists in {extract_to}.\n Skipping extraction.')
    return
  print(f'Extacting {zip_path} ...')
  start_time = time.time()
  with ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)
  elapsed_time = time.time() - start_time
  print(f'Extracted to {extract_to} in {elapsed_time:.2f} seconds.')

# get image zip files and extract it
download_file(dataset_url, dataset_zip_path)
extract_zip(dataset_zip_path, dataset_extract_path)

# get annotation zip file and extract it
download_file(annotations_url, annotations_zip_path)
extract_zip(annotations_zip_path, annotations_extract_path)

# Define a subclass of CoCo Detection

In [ ]:
from typing import Callable

import numpy as np
from PIL import Image
import torch
import torchvision


class CocoDetection(torchvision.datasets.CocoDetection):
  def __init__(self, img_directory: str, anno_file: str, transform: Callable=None):
    super().__init__(img_directory, anno_file)
    if transform is not None and not callable(transform):
      raise ValueError('transform must be callable')

    self.transform = transform
    self.classes = []
    self.coco_to_pred = []
    self.pred_to_coco = []

    for i, category_id in enumerate(sorted(self.coco.cats.keys())):
      self.classes.append(self.coco.cats[category_id]['name'])
      self.coco_to_pred[category_id] = i
      self.pred_to_coco[i] = category_id

  def __getitem__(self, idx: int):
    img, target = super().__getitem__(idx)
    img_id = self.ids[idx]

    # アノテーションデータに iscrowd がない or 値が0（密集領域ではない）データのみ保持
    target = [obj for obj in target if 'iscrowd' not in obj or obj['iscrowd'] == 0]

    # Albumentationsのデータ拡張処理において、BBoxが画像の外に出たり、
    # サイズがゼロのボックスになる場合の対策として
    # 幅と高さが0より大きいボックスのみを残すフィルター処理を行う
    target = [obj for obj in target if obj['bbox'][2] > 0 and obj['bbox'][3] > 0]

    classes = torch.tensor(
        [self.coco_to_pred[obj['cagtegory_id']] for obj in target],
        dtype=torch.int64
    )
    boxes = torch.tensor(
        [obj['bbox'] for obj in target],
        dtype=torch.float32
    )

    if boxes.numel() == 0:
      # 空のBBoxテンソルを生成：画像によっては物体が存在しない場合があるため、
      # このような画像で学習・推論時にエラーが出ないようにする
      boxes = torch.zeros((0, 4), dtype=torch.float32)
      classes = torch.zeros((0,), dtype=torch.int64)

    width, height = img.size
    if self.transform is not None:
      transformed = self.transform(
          image=np.array(img),
          bboxes=boxes.tolist(),
          classes=classes.tolist()
      )
      img = transformed['image']
      boxes = torch.tensor(transformed['classes'], dtype=torch.float32)
      classes = torch.tensor(transformed['classes'], dtype=torch.int64)
      if boxes.numel() == 0:
        return self.__getitem__((idx + 1) % len(self))

    # data形式を[xmin, ymin, width, height]から[xmin, ymin, xmax, ymax]に
    boxes[:, 2:] += boxes[:, :2]
    # bbox が画像の境界内に収まるようにクリップ
    boxes[:, ::2] = boxes[:, ::2].clamp(min=0, max=width)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(min=0, max=height)

    target = {
        'image_id': torch.tensor(img_id, dtype=torch.int64),
        'classes': classes,
        'boxes': boxes,
        'size': torch.tensor((width, height), dtype=torch.int64),
        'orig_size': torch.tensor((width, height), dtype=torch.int64),
        'orig_img': (
            torch.from_numpy(np.asarray(img))
            if isinstance(img, Image.Image) else torch.from_numpy(img)
            if isinstance(img, np.ndarray) else img.clone().detach()
        )
    }

    return img, target

  def to_coco_label(self, label: int):
    return self.pred_to_coco[label]

# Impliment DETR

DETR is composed with following components:
- Backbone Network (CNN): extracts feature map from the input image
- Transformer encoder/decoder: detects objects from the feature map
- Prediction head: generates bboxes and class labels from the outputs of other components

## Impliment ResNet-101

In [1]:
from torch import nn
from torchvision.ops.misc import FrozenBatchNorm2d
import torch

class ResNet101BottleneckBlock(nn.Module):
  """ResNet-101の残差ブロック
  ネットワークの基盤クラスである nn.Module を継承

  Attributes:
    conv1 (nn.Conv2d): 1x1畳み込み層（チャンネル圧縮）
    bn1 (FrozenBatchNorm2d): conv1のバッチ正規化
    conv2 (nn.Conv2d): 3x3
    bn2 (FrozenBatchNorm2d)
    conv3 (nn.Conv2d): 1x1
    bn3 (FrozenBatchNorm2d)
    relu (nn.ReLU): activation
    downsample (nn.Module): ダウンサンプリング用の畳み込み層
  """
  def __init__(
      self,
      in_channels: int,
      out_channels: int,
      stride: int=1,
      downsample: bool=False,
  ):
    """
    Args:
      in_channels (int): 入力特徴マップのチャネル数 (RGB画像なら3)
      out_channels (int): 畳み込み演算後の出力チャネル数
      stride (int): 3x3の畳み込み層ストライド
                    デフォルトは1、ダウンサンプリング時は2
      downsample (bool): Trueの場合、1x1畳み込みを使ってダウンサンプリングを行う
    """
    super().__init__()
    # bottleneckの中間チャネル数
    mid_channels = out_channels // 4

    # --- 1x1 畳み込み層(チャネル圧縮用) ---
    self.conv1 = nn.Conv2d(
        in_channels,  # 入力チャネル
        mid_channels,  # 出力チャネル
        kernel_size=1,  # 1x1畳み込み (空間サイズを変えずにチャネル数だけを変更)
        bias=False  # 後続のBatchNormが補うため、バイアスは使用しない
    )
    self.bn1 = FrozenBatchNorm2d(mid_channels)  # チャネルごとに平均・分散を標準化

    # --- 3x3 畳み込み層 (特徴抽出用) ---
    self.conv2 = nn.Conv2d(
        mid_channels,
        mid_channels,  # 入出力でチャネル数に変化なし
        kernel_size=3,  # 3x3 畳み込みで空間的な特徴を捉える
        stride=stride,  # 空間サイズを縮小するかどうか (1: そのまま, n: 1/nに縮小)
        padding=1,  # 出力サイズを入力と同じに保つためのゼロパディング
        bias=False,  # 後続のBatchNorm が代用
    )
    # conv2の出力に対するバッチ正規化(FrozenBNは学習中に統計値を固定するので、pre-trained modelのそれを引き継げる)
    self.bn2 = FrozenBatchNorm2d(mid_channels)

    # --- 1x1 畳み込み層(チャネル拡張用) ---
    self.conv3 = nn.Conv2d(
        mid_channels,
        out_channels,
        kernel_size=1,  # 1x1畳み込みで空間サイズを変えずにチャネル数を拡張
        bias=False
    )
    self.bn3 = FrozenBatchNorm2d(out_channels)

    # --- activation ---
    # inplace=Trueで入力テンソルに対して上書き処理を行う
    # メモリ使用量を抑えられるが、入力データ変更によって back prop. の時にエラーが出る可能性ある
    self.relu = nn.ReLU(inplace=True)

    self.downsample = None
    if downsample:
      # 残差接続の入力テンソルをメイン経路の出力形状に合わせるための補助処理
      self.downsample = nn.Sequential(
          nn.Conv2d(
              in_channels,  # 残差接続側のチャネル数
              out_channels,  # メイン経路の出力に揃える
              kernel_size=1,  # 1x1の畳み込み。空間サイズはそのまま、チャネル数だけを変換
              stride=stride,  # HxW -> H/stride x W/stride
              bias=False
          ),
          FrozenBatchNorm2d(out_channels)
      )

  def forward(self, x: torch.Tensor):
    identity = x  # 残差接続用の入力保存
    self.downsample is not None:
      identity = self.downsample(x)

    out1 = self.relu(self.bn1(self.conv1(x)))
    out2 = self.relu(self.bn2(self.conv2(out1)))
    out3 = self.bn3(self.conv3(out2))

    out3 += identity
    out = self.relu(out3)

    return out


class ResNet101(nn.Module):
  """ResNet-101 model
  Attributes
    conv1 (nn.Conv2d):
      - 入力画像をダウンサンプルし、チャネル数を拡張する7x7の畳み込み層
      - ストライド2、パディング3、バイアスなし
    bn1 (FrozenBatchNorm2d):
      - conv1のバッチ正規化層
    relu (nn.ReLU)
    max_pool (nn.MaxPool2d): 3x3 のMaxプーリング層(ストライド2、パディング1)
    layer1 (nn.Sequential): 第一層の残差ブロック(BottleNeck x 3)
    layer2 (nn.Sequential): 第二層の残差ブロック(BottleNeck x 4, ストライド2)
    layer3 (nn.Sequential): 第三層の残差ブロック(BottleNeck x 3, ストライド2)
    reduce_channels (nn.Conv2d): 出力チャネルサイズを2048->512に変換する1x1畳み込み層
    bn_reduce (FrozenBatchNorm2d): reduce_channels のバッチ正規化層
  """
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(
        3, 64, kernel_size=7, stride=2, padding=3, bias=False
    )
    self.bn1 = FrozenBatchNorm2d(64)
    self.relu = nn.ReLU(inplace=True)
    self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    # BottleNeck数をResNet-101の構成に合わせる
    self.layer1 = self._make_layer(64, 256, blocks=3, stride=1)
    self.layer2 = self._make_layer(256, 512, blocks=4, stride=2)
    self.layer3 = self._make_layer(512, 1024, blocks=23, stride=2)
    self.layer4 = self._make_layer(1024, 2048, blocks=3, stride=2)

    # 2048 -> 512 に変換する1x1畳み込み層
    self.reduce_channels = nn.Conv2d(2048, 512, kernel_size=1, bias=False)
    self.bn_reduce = FrozenBatchNorm2d(512)

  def _make_layer(self, in_channels, out_channels, blocks, stride):
    """複数のBottleNeckブロックを持つResNet-101の1層を構築
    Args:
      in_channels (int): 入力チャネル数 (前層の出力チャネル数に依存)
      out_channels (int)
      blocks (int): このレイヤー内に配置するBottleNeckブロックの数
      stride (int): 最初のBottleNeckブロックに適用するストライド (1or2)
    Returns:
      nn.Sequential: 指定された数のBottleNeckから構成されるSequential module
    """
    layers = [
        ResNet101BottleneckBlock(in_channels, out_channels, stride=stride, downsample=True)
    ]
    for _ in range(1, blocks):
      layers.append(ResNet101BottleneckBlock(out_channels, out_channels))

    return nn.Sequential(*layers)

  def forward(self, x: torch.Tensor):
    out = self.max_pool(self.relu(self.bn1(self.conv1(x))))
    out = self.layer1(out)
    layer2_output = self.layer2(out)
    layer3_output = self.layer3(layer2_output)
    layer4_output = self.layer4(layer3_output)

    # 出力次元をDETRの入力に合わせる 2048 -> 512
    layer4_output = self.relu(self.bn_reduce(self.reduce_channels(layer4_output)))

    return layer2_output, layer3_output, layer4_output


